In [24]:
import numpy as np
import gymnasium as gym

In [25]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", is_slippery=False)

env = env.unwrapped

In [26]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------
# Code here
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------
def policy_evaluation(env, policy, gamma=0.99, theta=1e-8):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    V = np.zeros(n_states)

    while True:

        delta = 0

        for s in range(n_states):

            v = V[s]
            new_v = 0

            # Bellman Expectation Equation
            for a, action_prob in enumerate(policy[s]):

                for prob, next_state, reward, done in env.P[s][a]:

                    new_v += action_prob * prob * (
                        reward + gamma * V[next_state] * (not done)
                    )

            V[s] = new_v
            delta = max(delta, abs(v - new_v))

        if delta < theta:
            break

    return V


In [27]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

# Code here
def policy_improvement(env, V, gamma=0.99):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.zeros((n_states, n_actions))

    for s in range(n_states):

        action_values = np.zeros(n_actions)

        for a in range(n_actions):

            for prob, next_state, reward, done in env.P[s][a]:

                action_values[a] += prob * (
                    reward + gamma * V[next_state] * (not done)
                )

        best_action = np.argmax(action_values)
        policy[s][best_action] = 1.0

    return policy

In [28]:
# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

# Code here
def policy_iteration(env, gamma=0.99, theta=1e-8):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # Initial Random Policy
    policy = np.ones((n_states, n_actions)) / n_actions

    # Save initial policy
    initial_policy = policy.copy()

    # Evaluate initial policy
    initial_value_function = policy_evaluation(
        env,
        initial_policy,
        gamma,
        theta
    )

    iterations = 0

    while True:

        # Policy Evaluation
        V = policy_evaluation(env, policy, gamma, theta)

        # Policy Improvement
        new_policy = policy_improvement(env, V, gamma)

        iterations += 1

        if np.array_equal(policy, new_policy):
            break

        policy = new_policy

    return (
        policy,
        V,
        iterations,
        initial_policy,
        initial_value_function
    )


In [29]:
# -------------------------------------------------
# Display Functions
# -------------------------------------------------


def print_value_function(V):

    print(np.round(V.reshape(4,4),4))


def print_policy(policy, title):

    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)

    policy_grid = np.array(
        [action_symbols[a] for a in best_actions]
    ).reshape(4,4)

    print(f"\n{title}:")
    print(policy_grid)

In [30]:
import numpy as np
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------


optimal_policy, optimal_value_function, num_iterations, initial_policy, initial_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name: kolluru pujitha")
print("Register Number: 212223240074")

print("\nInitial State-Value Function:")
print_value_function(initial_value_function)

print_policy(initial_policy, "Initial Policy")

print("\nIterations to convergence:", num_iterations)

print("\nOptimal State-Value Function:")
print_value_function(optimal_value_function)

print_policy(optimal_policy, "Optimal Policy")

env.close()


Name: kolluru pujitha
Register Number: 212223240074

Initial State-Value Function:
[[0.0124 0.0104 0.0193 0.0095]
 [0.0148 0.     0.0389 0.    ]
 [0.0326 0.0843 0.1378 0.    ]
 [0.     0.1703 0.4336 0.    ]]

Initial Policy:
[['←' '←' '←' '←']
 ['←' '←' '←' '←']
 ['←' '←' '←' '←']
 ['←' '←' '←' '←']]

Iterations to convergence: 2

Optimal State-Value Function:
[[0.951  0.9606 0.9703 0.9606]
 [0.9606 0.     0.9801 0.    ]
 [0.9703 0.9801 0.99   0.    ]
 [0.     0.99   1.     0.    ]]

Optimal Policy:
[['↓' '→' '↓' '←']
 ['↓' '←' '↓' '←']
 ['→' '↓' '↓' '←']
 ['←' '→' '→' '←']]
